# 00 Repo Smoke Test
Hizli smoke testi: unified runner + standart output format.

In [ ]:
import subprocess
import sys
import re
import json
import time
from pathlib import Path

candidate_roots = [
    Path.cwd(),
    Path.cwd().parent,
    Path('/content/LLMComparison'),
    Path('/content/drive/MyDrive/LLMComparison'),
]

PROJECT_ROOT = next(
    (
        root
        for root in candidate_roots
        if (root / 'experiments').exists() and (root / 'src').exists()
    ),
    None,
)

if PROJECT_ROOT is None:
    raise FileNotFoundError('Project root not found.')

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# === Smoke Test Configuration ===
MODEL = 'qwen2-vl-2b'
PRESET = 'smoke_cpu'
DATASETS = ['hf_vqa_rad']
NUM_SAMPLES = 3
SEED = 42
MAX_FALLBACK_RATE = 1.0
STRICT_SAMPLE_VALIDATION = False
OUTPUT_DIR = PROJECT_ROOT / 'results'
RUN_NAME = 'smoke_cpu_qwen2_vqa'

print('🧪 Repo Smoke Test')
print(f'Model: {MODEL}')
print(f'Preset: {PRESET}')
print(f'Samples: {NUM_SAMPLES}')
print(f'Run: {RUN_NAME}')

command = [
    sys.executable,
    str(PROJECT_ROOT / 'experiments' / 'run_unified.py'),
    '--preset', PRESET,
    '--models', MODEL,
    '--datasets', *DATASETS,
    '--num-samples', str(NUM_SAMPLES),
    '--seed', str(SEED),
    '--max-fallback-rate', str(MAX_FALLBACK_RATE),
    '--skip-inaccessible',
    '--output-dir', str(OUTPUT_DIR),
    '--run-name', RUN_NAME,
    '--debug',
]

if STRICT_SAMPLE_VALIDATION:
    command.append('--strict-sample-validation')

print('\nRunning smoke test...')
start_ts = time.time()

process = subprocess.Popen(
    command,
    cwd=PROJECT_ROOT,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

for line in process.stdout:
    print(line.rstrip())

return_code = process.wait()
elapsed_min = (time.time() - start_ts) / 60.0

if return_code == 0:
    print(f'\n✅ PASS - Smoke test completed in {elapsed_min:.1f}m')
else:
    print(f'\n❌ FAIL - Smoke test failed with code {return_code}')
    raise RuntimeError(f'Smoke test failed')

In [ ]:
import json
import pandas as pd
from pathlib import Path

run_dir = OUTPUT_DIR / RUN_NAME
required_files = [
    'config_snapshot.json',
    'environment.json',
    'predictions.jsonl',
    'sample_metrics.jsonl',
    'aggregate_metrics.json',
    'stats.json',
    'errors.jsonl',
]

print('=== SMOKE TEST VERIFICATION ===\n')
print(f'Run directory: {run_dir}')
print(f'Exists: {run_dir.exists()}\n')

if run_dir.exists():
    actual_files = sorted([p.name for p in run_dir.glob('*')])
    print('Output files:')
    for fname in actual_files:
        fpath = run_dir / fname
        fsize = fpath.stat().st_size
        print(f'  ✓ {fname:30s} ({fsize:8d} bytes)')
    
    missing = set(required_files) - set(actual_files)
    if missing:
        print(f'\n⚠️  Missing files: {", ".join(missing)}')
    else:
        print(f'\n✅ All required files present')
    
    # === Verify output structure ===
    print('\n=== OUTPUT STRUCTURE CHECKS ===\n')
    
    if (run_dir / 'aggregate_metrics.json').exists():
        agg = json.loads((run_dir / 'aggregate_metrics.json').read_text())
        print(f'aggregate_metrics.json:')
        print(f'  Models: {list(agg.get("metrics", {}).keys())}')
        print(f'  Metrics per model: {len(list(agg.get("metrics", {}).values())[0]) if agg.get("metrics") else 0}')
    
    if (run_dir / 'stats.json').exists():
        stats = json.loads((run_dir / 'stats.json').read_text())
        print(f'\nstats.json:')
        print(f'  Statistics keys: {list(stats.get("statistics", {}).keys())}')
        if '_meta_fallback_rates' in stats.get('statistics', {}):
            print(f'  ✓ Fallback rates present (metadata)')
    
    if (run_dir / 'sample_metrics.jsonl').exists():
        samples = [json.loads(line) for line in (run_dir / 'sample_metrics.jsonl').read_text().splitlines() if line.strip()]
        print(f'\nsample_metrics.jsonl:')
        print(f'  Total rows: {len(samples)}')
        if samples:
            print(f'  Keys per row: {list(samples[0].keys())}')
            fallback_cols = [k for k in samples[0].keys() if k.endswith('_fallback_used')]
            if fallback_cols:
                print(f'  ✓ Fallback columns present: {fallback_cols}')
    
    if (run_dir / 'errors.jsonl').exists():
        errors = [json.loads(line) for line in (run_dir / 'errors.jsonl').read_text().splitlines() if line.strip()]
        if errors:
            error_df = pd.DataFrame(errors)
            print(f'\nerrors.jsonl:')
            print(f'  Error types: {error_df.groupby("error_type").size().to_dict()}')
        else:
            print(f'\nerrors.jsonl: (empty - no errors)')
    
    print('\n✅ SMOKE TEST VERIFICATION COMPLETE')
else:
    print('❌ Run directory not found!')